In [1]:
import numpy as np
from sklearn.model_selection import train_test_split
from MuyGPyS._test.sampler import print_results
from MuyGPyS.neighbors import NN_Wrapper
from MuyGPyS.gp import MuyGPS
from MuyGPyS.gp.deformation import Isotropy, l2
from MuyGPyS.gp.hyperparameter import AnalyticScale, Parameter
from MuyGPyS.gp.kernels import Matern
from MuyGPyS.gp.noise import HomoscedasticNoise
from MuyGPyS.optimize import Bayes_optimize, L_BFGS_B_optimize
from MuyGPyS.optimize.batch import sample_batch
from MuyGPyS.optimize.loss import lool_fn

# Import and setup data

In [2]:
aqi_data = np.genfromtxt("biggest_day_aqi_coords_scaled.csv", delimiter=',', skip_header=1)
print(aqi_data[:10])

[[ 0.27128141  0.75872801 53.        ]
 [ 0.33042753  0.78042228 47.        ]
 [ 0.36178224  0.76125781 69.        ]
 [ 0.35172346  0.7786855  69.        ]
 [ 0.31374196  0.77694373 48.        ]
 [ 0.2842055   0.76718129 52.        ]
 [ 0.34535027  0.77844838 65.        ]
 [ 0.33611686  0.76985651 74.        ]
 [ 0.35287642  0.76437478 63.        ]
 [ 0.36015016  0.76916901 45.        ]]


In [3]:
#randomly split the data into features and responces, and training and testing paritions.
aqi_x, aqi_y = aqi_data[:, :-1], aqi_data[:, -1]
aqi_x_train, aqi_x_test, aqi_y_train, aqi_y_test = train_test_split(aqi_x, aqi_y, test_size=0.1, random_state=24)
#the below was created to try and istimate a mean functin.
# aqi_y_mean = aqi_y_train.mean()
# aqi_y_train = aqi_y_train - aqi_y_mean
# aqi_y_test = aqi_y_test - aqi_y_mean
print(aqi_x_train[:10])
print(aqi_y_train[:10])

[[0.53471228 0.50116004]
 [0.45663329 0.876782  ]
 [0.51078227 0.91802638]
 [0.5564572  0.72190225]
 [0.54847353 0.94335291]
 [1.         0.13357091]
 [0.62777993 0.61365714]
 [0.58149488 0.70675474]
 [0.48818196 0.90427799]
 [0.54840132 0.59572094]]
[36. 50. 45. 32. 42. 24. 49. 64. 42. 17.]


# Nearest Neighbor and Batches

In [4]:
#setup nearest nieghbors
nn_count = 30
nbrs_lookup = NN_Wrapper(aqi_x_train, nn_count, nn_method="exact", algorithm="ball_tree")


In [5]:
#for the sake of making use of batches, will use a batch value of 1500
batch_count = 1500
batch_indices, batch_nn_indices = sample_batch(
    nbrs_lookup, batch_count, len(aqi_x_train)
)

# setting and optimizing hyperparamters

In [6]:
aqi_muygps = MuyGPS(
    kernel=Matern(
        smoothness=Parameter("log_sample", (0.01, 4.0)),
        deformation=Isotropy(
            l2,
            length_scale=Parameter("log_sample", (0.01, 0.3)),
        ),
    ),
    noise=HomoscedasticNoise("sample", (1e-7, 1e-3)),
    scale=AnalyticScale(),
)

In [7]:
# #crosswise distance: distances between a point and each of it's neighbors
# #pairwise distance: the distances between all of a point's neighbors and each other. for the purpose of finding which points contribute together and should share credit.

# batch_ys and batch_nn_ys are the responces to the training points, which we need to grab.
(
    batch_crosswise_dists,
    batch_pairwise_dists,
    batch_ys,
    batch_nn_ys,
) = aqi_muygps.make_train_tensors(
    batch_indices,
    batch_nn_indices,
    aqi_x_train,
    aqi_y_train,
)
#there are the distance tensors above which store the tensors from each batch point to it's neighbors
#as well as, for each batch point, between each of the neighbors and each other.

# #these get the covariances for the pairwise and crosswise distances.
kcross = aqi_muygps.kernel(batch_crosswise_dists)
kin = aqi_muygps.kernel(batch_pairwise_dists)
#and these are the kernel tensors, which store the covariance of how these distances translate into correlation.

#you wnat both the batch point's relations, and the relations between it's nieghbors,
#so that you can find out which neighbors should "share" their contriobution between them because they're similar.
#you need the pairwise kernel, in addition to the distances, in order to find out which
#points to attribute with being used for hte answer.

In [8]:
#optimize the parameters
aqi_muygps_optimized = Bayes_optimize(
    aqi_muygps,
    batch_ys,
    batch_nn_ys,
    batch_crosswise_dists,
    batch_pairwise_dists,
    loss_fn=lool_fn,
    verbose=True,
    random_state=24,
    init_points=15,
    n_iter=25,
)

# aqi_muygps_optimized = L_BFGS_B_optimize(
#     aqi_muygps,
#     batch_ys,
#     batch_nn_ys,
#     batch_crosswise_dists,
#     batch_pairwise_dists,
#     loss_fn=lool_fn,
# )

parameters to be optimized: ['length_scale', 'smoothness', 'noise']
bounds: [[1.e-02 1.e+00]
 [1.e-02 4.e+00]
 [1.e-07 1.e+00]]
initial x0: [0.16281672 2.83355358 0.486231  ]
|   iter    |  target   | length... | smooth... |   noise   |
-------------------------------------------------------------
| 1         | -22417.10 | 0.1628167 | 2.8335535 | 0.4862310 |
| 2         | -15605.53 | 0.9604171 | 2.8010530 | 0.9998672 |
| 3         | -16129.93 | 0.2278666 | 1.4506148 | 0.7398410 |
| 4         | -80976.58 | 0.9964911 | 1.2722244 | 0.1365446 |
| 5         | -29336.93 | 0.3901402 | 1.2888719 | 0.3664148 |
| 6         | -25234.90 | 0.7125550 | 3.6015682 | 0.5341154 |
| 7         | -22108.99 | 0.2548208 | 2.6905081 | 0.5617291 |
| 8         | -17420.68 | 0.5471342 | 3.5748559 | 0.8427795 |
| 9         | -19557.07 | 0.3129524 | 2.5283674 | 0.6802388 |
| 10        | -16305.62 | 0.9707232 | 3.5753329 | 0.9424258 |
| 11        | -52395.88 | 0.6458032 | 2.4624440 | 0.2276833 |
| 12        | -1730

In [9]:
#also optimize the scale
aqi_muygps_optimized = aqi_muygps_optimized.optimize_scale(
    batch_pairwise_dists,
    batch_nn_ys
)

# Inference

In [10]:
test_count = aqi_x_test.shape[0]
test_indices = np.arange(test_count)
test_nn_indices, _ = nbrs_lookup.get_nns(aqi_x_test)

In [11]:
(
    test_crosswise_dists,
    test_pairwise_dists,
    test_nn_ys,
) = aqi_muygps.make_predict_tensors(
    test_indices,
    test_nn_indices,
    aqi_x_test,
    aqi_x_train,
    aqi_y_train,
)

kcross = aqi_muygps_optimized.kernel(test_crosswise_dists)
kin = aqi_muygps_optimized.kernel(test_pairwise_dists)

In [12]:
predictions = aqi_muygps_optimized.posterior_mean(kin, kcross, test_nn_ys)
variances = aqi_muygps_optimized.posterior_variance(kin, kcross)
#confidence interval means within 1.96 of the standard of deviation for that point.
confidence_intervals = np.sqrt(variances) * 1.96
#coverage is the proportion of posterior means (guesses) that differ from the true response by no more than the confidence interval size.
coverage = np.count_nonzero(np.abs(aqi_y_test - predictions) < confidence_intervals) / test_count

In [13]:
print_results(
    aqi_y_test, ("optimized", aqi_muygps_optimized, predictions, variances, confidence_intervals, coverage)
)

name,smoothness,length scale,noise variance,variance scale,rmse,mean variance,mean confidence interval,coverage
optimized,0.010000,1.000000,0.000000,1010.425442,16.735717,905.964870,58.993431,1.000000
